# Model 3 — XGBoost for FD001 Remaining Useful Life

This notebook predicts how many cycles remain before a turbofan engine fails. It builds a clear XGBoost baseline, then tests a stronger version using capped targets and causal degradation features. The comparison is original to this project and uses engine-level validation to prevent leakage.

## 1. Setup

The code first finds the repository root so the notebook runs from either VS Code or the command line. Generated figures, predictions, metrics, and the final model are saved in the project folders.

In [ ]:
from pathlib import Path
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cmapss_rul.data import (
    last_cycle_rows,
    load_fd001,
    make_validation_subset,
    split_by_engine,
)
from cmapss_rul.evaluation import regression_metrics
from cmapss_rul.features import (
    engineer_features,
    nonconstant_sensor_columns,
    select_sensors,
)
from cmapss_rul.modeling import (
    baseline_parameters,
    improved_parameters,
    resolve_device,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'cmapss' / 'CMaps'
FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'
MODEL_DIR = PROJECT_ROOT / 'models'
REPORT_DIR = PROJECT_ROOT / 'reports'
for directory in (FIGURE_DIR, MODEL_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 12)
RANDOM_STATE = 42
RUL_CAP = 125
print(f'Project root: {PROJECT_ROOT}')

## 2. Load and understand FD001

FD001 contains one operating condition and one fault mode. Every row is one engine cycle with three settings and 21 sensor readings. Training trajectories continue to failure; test trajectories stop earlier and have one official RUL label per engine.

In [ ]:
train_df, test_df, official_test = load_fd001(DATA_DIR)

dataset_summary = pd.DataFrame({
    'rows': [len(train_df), len(test_df)],
    'engines': [train_df['unit_id'].nunique(), test_df['unit_id'].nunique()],
    'minimum_cycle': [train_df['cycle'].min(), test_df['cycle'].min()],
    'maximum_cycle': [train_df['cycle'].max(), test_df['cycle'].max()],
}, index=['train', 'test'])

print(f'Training shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
print(f'Official test labels: {len(official_test)}')
display(dataset_summary)
display(train_df.head())

## 3. Visual exploration

Engine lifetime varies substantially, so a random row split would mix readings from the same engine across train and validation data. The sensor plot uses normalized values only for visualization, making different measurement scales comparable.

In [ ]:
engine_lifetimes = train_df.groupby('unit_id')['cycle'].max()
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(engine_lifetimes, bins=18, kde=True, color='#2A6F97', ax=ax)
ax.axvline(engine_lifetimes.mean(), color='#D1495B', linestyle='--', label=f'Mean = {engine_lifetimes.mean():.1f}')
ax.set(title='FD001 engine lifetime distribution', xlabel='Lifetime (cycles)', ylabel='Number of engines')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'xgboost_fd001_lifetimes.png', dpi=160, bbox_inches='tight')
plt.show()
display(engine_lifetimes.describe().to_frame('cycles').T.round(2))

In [ ]:
example_engine = train_df.loc[train_df['unit_id'].eq(1)].copy()
visual_sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_12', 'sensor_15', 'sensor_21']
normalized = example_engine[visual_sensors].apply(lambda column: (column - column.mean()) / column.std())
normalized['cycle'] = example_engine['cycle'].to_numpy()
sensor_long = normalized.melt('cycle', var_name='sensor', value_name='standardized_value')

fig, ax = plt.subplots(figsize=(11, 6))
sns.lineplot(data=sensor_long, x='cycle', y='standardized_value', hue='sensor', linewidth=1.4, ax=ax)
ax.set(title='Engine 1: sensor trajectories as failure approaches', ylabel='Standardized sensor value')
ax.legend(ncol=2, title='Sensor')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'xgboost_fd001_sensor_trends.png', dpi=160, bbox_inches='tight')
plt.show()

## 4. Leakage-safe validation

Complete engines are assigned to either training or validation. Validation engines are then truncated before failure at deterministic random cutoffs, creating realistic partial histories like the official test set.

In [ ]:
train_split, validation_full = split_by_engine(
    train_df, validation_size=0.20, random_state=RANDOM_STATE
)
validation_history, validation_snapshots = make_validation_subset(
    validation_full, min_rul=10, max_rul=100, random_state=RANDOM_STATE
)

train_engines = set(train_split['unit_id'])
validation_engines = set(validation_history['unit_id'])
assert train_engines.isdisjoint(validation_engines)
assert len(validation_snapshots) == len(validation_engines)

device = resolve_device(prefer_gpu=True)
print(f'XGBoost version: {xgb.__version__}')
print(f'Training device selected: {device}')
print(f'Training engines: {len(train_engines)} | Validation engines: {len(validation_engines)}')
display(validation_snapshots[['unit_id', 'cycle', 'rul']].head())

## 5. Baseline XGBoost

The baseline uses cycle, operating settings, and all non-constant raw sensors. It has no rolling degradation features and learns the uncapped training target.

In [ ]:
raw_sensors = nonconstant_sensor_columns(train_split)
baseline_columns = ['cycle', 'setting_1', 'setting_2', 'setting_3', *raw_sensors]

baseline_model = xgb.XGBRegressor(**baseline_parameters(device))
baseline_start = time.perf_counter()
baseline_model.fit(train_split[baseline_columns], train_split['rul'], verbose=False)
baseline_train_seconds = time.perf_counter() - baseline_start

baseline_predict_start = time.perf_counter()
baseline_predictions = np.clip(
    baseline_model.predict(validation_snapshots[baseline_columns]), 0.0, None
)
baseline_predict_seconds = time.perf_counter() - baseline_predict_start
baseline_metrics = regression_metrics(validation_snapshots['rul'], baseline_predictions)
display(pd.DataFrame([baseline_metrics], index=['Baseline']).round(3))

## 6. Improved XGBoost

The improved model caps early-life RUL at 125, selects informative sensors using training engines only, and adds causal differences, rolling means, rolling variation, and degradation slopes. Early stopping limits unnecessary trees.

In [ ]:
selection_frame = train_split.assign(rul=train_split['rul'].clip(upper=RUL_CAP))
selected_sensors = select_sensors(selection_frame, top_n=8)
print('Training-only selected sensors:', selected_sensors)

X_train_improved = engineer_features(train_split, selected_sensors, windows=(5, 15))
X_valid_history = engineer_features(validation_history, selected_sensors, windows=(5, 15))
validation_last_indices = validation_history.groupby('unit_id')['cycle'].idxmax()
X_valid_last = X_valid_history.loc[validation_last_indices].reset_index(drop=True)
y_train_improved = train_split['rul'].clip(upper=RUL_CAP)
y_valid_history = validation_history['rul'].clip(upper=RUL_CAP)

improved_model = xgb.XGBRegressor(**improved_parameters(device))
improved_start = time.perf_counter()
improved_model.fit(
    X_train_improved,
    y_train_improved,
    eval_set=[(X_valid_history, y_valid_history)],
    verbose=False,
)
improved_train_seconds = time.perf_counter() - improved_start

improved_predict_start = time.perf_counter()
improved_predictions = np.clip(improved_model.predict(X_valid_last), 0.0, None)
improved_predict_seconds = time.perf_counter() - improved_predict_start
improved_metrics = regression_metrics(validation_snapshots['rul'], improved_predictions)
print(f'Best boosting round: {improved_model.best_iteration}')
display(pd.DataFrame([improved_metrics], index=['Improved']).round(3))

## 7. Compare the two XGBoost versions

Lower RMSE, MAE, and NASA score are better; higher R² is better. The comparison uses identical validation engines and cutoff points.

In [ ]:
metric_rows = [
    {
        'model': 'XGBoost baseline', 'split': 'validation', 'device': device,
        'train_seconds': baseline_train_seconds, 'predict_seconds': baseline_predict_seconds,
        **baseline_metrics,
    },
    {
        'model': 'XGBoost improved', 'split': 'validation', 'device': device,
        'train_seconds': improved_train_seconds, 'predict_seconds': improved_predict_seconds,
        **improved_metrics,
    },
]
comparison = pd.DataFrame(metric_rows)
display(comparison.round(3))

plot_metrics = comparison.melt(
    id_vars='model', value_vars=['rmse', 'mae'], var_name='metric', value_name='value'
)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=plot_metrics, x='metric', y='value', hue='model', palette=['#7A7A7A', '#2A9D8F'], ax=ax)
ax.set(title='Validation error: baseline vs improved XGBoost', xlabel='', ylabel='Cycles (lower is better)')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'xgboost_fd001_metric_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
validation_results = validation_snapshots[['unit_id', 'rul']].rename(columns={'rul': 'true_rul'}).copy()
validation_results['baseline_prediction'] = baseline_predictions
validation_results['improved_prediction'] = improved_predictions
validation_results['improved_residual'] = validation_results['improved_prediction'] - validation_results['true_rul']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
limit = max(validation_results['true_rul'].max(), validation_results['improved_prediction'].max()) + 5
sns.scatterplot(data=validation_results, x='true_rul', y='improved_prediction', s=65, color='#2A9D8F', ax=axes[0])
axes[0].plot([0, limit], [0, limit], '--', color='#D1495B', label='Perfect prediction')
axes[0].set(title='Improved model: true vs predicted RUL', xlabel='True RUL', ylabel='Predicted RUL', xlim=(0, limit), ylim=(0, limit))
axes[0].legend()
sns.histplot(validation_results['improved_residual'], bins=12, kde=True, color='#457B9D', ax=axes[1])
axes[1].axvline(0, color='#D1495B', linestyle='--')
axes[1].set(title='Improved-model residuals', xlabel='Prediction − true RUL')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'xgboost_fd001_validation_diagnostics.png', dpi=160, bbox_inches='tight')
plt.show()

## 8. Train the final model and evaluate the official test set

After the method is frozen, feature selection is learned again from all training engines. The final number of trees comes from early stopping on the validation experiment, and official predictions use only each test engine's observed history.

In [ ]:
full_selection_frame = train_df.assign(rul=train_df['rul'].clip(upper=RUL_CAP))
final_sensors = select_sensors(full_selection_frame, top_n=8)
X_full_train = engineer_features(train_df, final_sensors, windows=(5, 15))
X_full_test = engineer_features(test_df, final_sensors, windows=(5, 15))
test_last_indices = test_df.groupby('unit_id')['cycle'].idxmax()
X_test_last = X_full_test.loc[test_last_indices].reset_index(drop=True)

final_params = improved_parameters(device)
final_params.pop('early_stopping_rounds')
final_params['n_estimators'] = max(1, improved_model.best_iteration + 1)
final_model = xgb.XGBRegressor(**final_params)
final_start = time.perf_counter()
final_model.fit(X_full_train, train_df['rul'].clip(upper=RUL_CAP), verbose=False)
final_train_seconds = time.perf_counter() - final_start

test_predict_start = time.perf_counter()
test_predictions = np.clip(final_model.predict(X_test_last), 0.0, None)
test_predict_seconds = time.perf_counter() - test_predict_start
official_metrics = regression_metrics(official_test['rul'], test_predictions)
metric_rows.append({
    'model': 'XGBoost improved', 'split': 'official_test', 'device': device,
    'train_seconds': final_train_seconds, 'predict_seconds': test_predict_seconds,
    **official_metrics,
})
display(pd.DataFrame([official_metrics], index=['Official FD001 test']).round(3))

In [ ]:
test_results = pd.DataFrame({
    'unit_id': official_test['unit_id'].astype(int),
    'true_rul': official_test['rul'],
    'predicted_rul': test_predictions,
})
test_results['residual'] = test_results['predicted_rul'] - test_results['true_rul']

fig, ax = plt.subplots(figsize=(9, 6))
limit = max(test_results['true_rul'].max(), test_results['predicted_rul'].max()) + 5
sns.scatterplot(data=test_results, x='true_rul', y='predicted_rul', hue='residual', palette='coolwarm', s=58, ax=ax)
ax.plot([0, limit], [0, limit], '--', color='black', linewidth=1.2, label='Perfect prediction')
ax.set(title='Official FD001 test predictions', xlabel='True RUL (cycles)', ylabel='Predicted RUL (cycles)', xlim=(0, limit), ylim=(0, limit))
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'xgboost_fd001_predictions.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
importance = pd.Series(final_model.feature_importances_, index=X_full_train.columns).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=importance.values, y=importance.index, color='#2A9D8F', ax=ax)
ax.set(title='Top 15 XGBoost feature importances', xlabel='Gain-based importance', ylabel='Feature')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'xgboost_fd001_feature_importance.png', dpi=160, bbox_inches='tight')
plt.show()
display(importance.to_frame('importance').round(4))

## 9. Save reproducible results

The model JSON and figures are generated locally and ignored by Git. Compact metrics and prediction CSV files are saved for the report and team comparison.

In [ ]:
metrics_table = pd.DataFrame(metric_rows)
metrics_path = REPORT_DIR / 'xgboost_fd001_metrics.csv'
predictions_path = REPORT_DIR / 'xgboost_fd001_predictions.csv'
model_path = MODEL_DIR / 'xgboost_fd001.json'

metrics_table.to_csv(metrics_path, index=False)
test_results.to_csv(predictions_path, index=False)
final_model.save_model(model_path)

validation_change = baseline_metrics['rmse'] - improved_metrics['rmse']
direction = 'improved' if validation_change > 0 else 'did not improve'
print(f'The engineered model {direction} validation RMSE by {abs(validation_change):.2f} cycles.')
print(f'Official FD001 RMSE: {official_metrics["rmse"]:.2f} cycles')
print(f'Saved metrics: {metrics_path}')
print(f'Saved predictions: {predictions_path}')
print(f'Saved model: {model_path}')
display(metrics_table.round(3))